# P03 — Auditoría del corpus
## (a) Paso del protocolo y objetivo
**Paso P03**: auditar el corpus antes de partirlo: (1) grupos de paráfrasis que mezclan intenciones, (2) **casi-duplicados** dentro de una misma intención (Jaccard de tokens o Levenshtein normalizada ≥ 0,90), (3) mismo texto con intenciones distintas (ambigüedad) y (4) **desbalance** por intención y categoría. Sustenta el **OE2**: un corpus mal auditado produce métricas engañosas.
**Origen:** auditoría en vivo sobre el corpus **Sintético**; para los lotes **Reales** solo se muestran conteos.

## (b) Código del repositorio que lo implementa
`audit_corpus.py` (P03/P04). Entradas: `corpus/corpus_metadata.csv` y `configs/jerga_local.csv`. Salidas: `corpus/corpus_audit.csv` y `logs/audit_report.txt`. Con `--apply` fusiona los grupos casi-duplicados (en este cuaderno **no se escribe nada**: solo se reutilizan sus funciones de similitud, en memoria).

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("03"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,audit_corpus.py,"P03, P04",P03 / P04 — Auditoría del Corpus MPSR-Bot y control de fuga por paráfrasis.,corpus/corpus_metadata.csv,"logs/audit_report.txt, corpus/corpus_audit.csv, corpus_summary.json",python scripts/audit_corpus.py # solo reporta,No,Lógica recalculada (etapa 4)


## (c) Celdas de código
### 1. Auditoría recalculada con las mismas funciones del script (Sintético)

In [3]:
sys.path.insert(0, str(BASE / "scripts"))
import audit_corpus as A                       # solo se usan sus funciones; no se ejecuta main()
from itertools import combinations
import common

cm = pd.read_csv(BASE / "corpus/corpus_metadata.csv", dtype=str, keep_default_na=False, encoding="utf-8")
jerga = common.load_jerga()
cm["norm_text"] = cm["text"].map(lambda t: common.normalize(t, jerga))

mezclan = [g for g, d in cm.groupby("base_phrase_id") if d["intent"].nunique() > 1]
pares = []
for intent, g in cm.groupby("intent"):
    filas = list(g[["utterance_id", "norm_text", "base_phrase_id"]].itertuples(index=False))
    for a, b in combinations(filas, 2):
        sim, metrica = A.similarity(a.norm_text, b.norm_text)
        if sim >= A.THRESHOLD:
            pares.append((a.utterance_id, b.utterance_id, a.base_phrase_id == b.base_phrase_id, round(sim, 3), metrica))
ambiguos = [t for t, g in cm.groupby("norm_text") if g["intent"].nunique() > 1]
rotulo(ORIGEN_SINTETICO, "auditoría del corpus v3 recalculada (en memoria)")
print("Grupos con varias intenciones:", len(mezclan), "| pares casi-duplicados:", len(pares), "| de ellos en OTRO grupo:", sum(not p[2] for p in pares), "| textos ambiguos entre intenciones:", len(ambiguos))
display(pd.DataFrame(pares, columns=["frase A", "frase B", "mismo grupo", "similitud", "métrica"]))

guardado = pd.read_csv(BASE / "corpus/corpus_audit.csv", dtype=str, keep_default_na=False, encoding="utf-8")
print("Registro guardado corpus/corpus_audit.csv:", guardado["issue_type"].value_counts().to_dict())
comparar("P03", "pares casi-duplicados", len(pares), len(guardado[guardado["issue_type"].str.startswith("casi_duplicado")]), "corpus/corpus_audit.csv", tol=0)
comparar("P03", "grupos que mezclan intenciones", len(mezclan), 0, "logs/audit_report.txt", tol=0)

▌ORIGEN: Sintético — auditoría del corpus v3 recalculada (en memoria)
Grupos con varias intenciones: 0 | pares casi-duplicados: 2 | de ellos en OTRO grupo: 2 | textos ambiguos entre intenciones: 0


,frase A,frase B,mismo grupo,similitud,métrica
0,U0018,U0020,False,0.964,levenshtein
1,U0002,U0008,False,0.953,levenshtein


Registro guardado corpus/corpus_audit.csv: {'casi_duplicado_otro_grupo': 2}


### 2. Desbalance y distribución (Sintético)

In [4]:
por_int = cm["intent"].value_counts()
grupos_int = cm.groupby("intent")["base_phrase_id"].nunique()
desbalance = por_int.max() / por_int.min()
rotulo(ORIGEN_SINTETICO, "distribución del corpus v3")
print(f"Frases por intención: mínimo {por_int.min()} · máximo {por_int.max()} · desbalance máx/mín = {desbalance:.2f}")
print("Intenciones con menos de 3 grupos de frase base (no se pueden repartir en train/val/test):", list(grupos_int[grupos_int < 3].index) or "ninguna")
display(cm["category"].value_counts().rename("frases").to_frame().T)
reporte = (BASE / "logs/audit_report.txt").read_text(encoding="utf-8")
m = re.search(r"Desbalance \(máx/mín utterances por intención\): ([\d.]+)", reporte)
comparar("P03", "desbalance máx/mín", round(desbalance, 2), float(m.group(1)), "logs/audit_report.txt", tol=0.005)
comparar("P03", "intenciones con menos de 3 grupos", int((grupos_int < 3).sum()), 0, "logs/audit_report.txt", tol=0)
print("\nPrimeras líneas del reporte guardado (logs/audit_report.txt):")
print("\n".join(reporte.splitlines()[:9]))

▌ORIGEN: Sintético — distribución del corpus v3
Frases por intención: mínimo 12 · máximo 21 · desbalance máx/mín = 1.75
Intenciones con menos de 3 grupos de frase base (no se pueden repartir en train/val/test): ninguna


category,Interacción conversacional,Trámites documentarios,Licencias y autorizaciones,Tributos y pagos municipales,Información general,Defensa civil y seguridad ciudadana,Servicios públicos,Registro civil,Reclamos y quejas
frases,147,84,81,72,72,66,66,60,60



Primeras líneas del reporte guardado (logs/audit_report.txt):
REPORTE DE AUDITORÍA — Corpus MPSR-Bot (P03/P04)
Archivo: C:\Users\Luis Mario\Documents\tesis2\mpsr-chatbot\mpsr-chatbot\corpus\corpus_metadata.csv
SHA-256: 9e10876af2eea2fa1a4efb711ed478cda5f51daaffe7a4d60ea7af7a75ffb462
Utterances: 708 | Intenciones: 54 (esperadas 50) | Categorías: 9 (esperadas 9) | Grupos base_phrase_id: 236
Umbral casi-duplicados: 0.9 (Jaccard de tokens o Levenshtein normalizada, misma intención)

Incidencias detectadas:
  casi_duplicado_otro_grupo: 2
  Pares casi-duplicados: 2


### 3. Lotes reales: cobertura por intención (Real, solo conteos)
Para cada lote real se comprueba que cada intención tenga al menos 3 frases (condición de G1 para el lote 1).

In [5]:
rotulo(ORIGEN_REAL, "conteos de frases reales validadas por intención (sin frases)")
for lote, d in ag["real_por_intencion"].items():
    s = pd.Series(d)
    print(f"{lote}: {int(s.sum())} frases en {len(s)} intenciones · mínimo {int(s.min())} · máximo {int(s.max())} · intenciones con menos de 3 frases: {int((s < 3).sum())}")
comparar("P03", "lote 2: intenciones con menos de 3 frases", int((pd.Series(ag["real_por_intencion"]["lote2_final"]) < 3).sum()), 0, "ingesta_lote2_cifras.md", tol=0)

▌ORIGEN: Real — conteos de frases reales validadas por intención (sin frases)
lote1_final: 189 frases en 54 intenciones · mínimo 3 · máximo 8 · intenciones con menos de 3 frases: 0
lote2_final: 280 frases en 54 intenciones · mínimo 5 · máximo 15 · intenciones con menos de 3 frases: 0


In [6]:
cierre("P03 Auditoría")

P03 Auditoría — recalculado frente a reportado: 5 cifras · coinciden: 5 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,P03,pares casi-duplicados,2.00,2.00,corpus/corpus_audit.csv,Sí
1,P03,grupos que mezclan intenciones,0.00,0.00,logs/audit_report.txt,Sí
2,P03,desbalance máx/mín,1.75,1.75,logs/audit_report.txt,Sí
3,P03,intenciones con menos de 3 grupos,0.00,0.00,logs/audit_report.txt,Sí
4,P03,lote 2: intenciones con menos de 3 frases,0.00,0.00,ingesta_lote2_cifras.md,Sí


DISCREPANCIAS (recalculado ≠ reportado): ninguna
Rasa disponible: False · paquete privado: True


## (d) Cómo leer el resultado
- La auditoría del corpus v3 detecta **2 pares casi-duplicados en grupos distintos** de la misma intención (similitud 0,95–0,96), que `audit_corpus.py --apply` fusiona en un mismo `base_phrase_id` para que no caigan en particiones distintas (ese es el vínculo con P04). No hay grupos que mezclen intenciones ni textos ambiguos.
- El desbalance máx/mín es de **1,75** (de 12 a 21 frases por intención) y todas las intenciones tienen al menos 4 grupos de paráfrasis.
- Las cifras recalculadas coinciden con el reporte guardado; si alguna dijera «NO», sería una discrepancia a reportar, no a corregir.

## Limitaciones
- La similitud mide forma, no significado: dos frases con otras palabras pero la misma intención no se detectan como paráfrasis.
- La auditoría es del corpus **sintético**; los lotes reales se auditan en la ingesta (cuaderno 11) y con la exclusión de duplicados exactos (cuadernos 04 y 05).